**`03_show_land_and_building_values_3d`**

This notebook constructs an interactive 3D map of land and building values.

* **Height & Color**: Represent value per unit area (footprint or parcel). Low values are short/blue; high values are tall/red.
* **Volume**: Proportional to the total asset value (USD) since Volume = Area * Height.
* **Stacking**: Building footprints are stacked on top of their parcel (if multiple: the one with the largest overlap).
* **Basemap**: Change using `--basemap` (`satellite`, `osm`, `positron`, `dark_matter`, or `topo`).

### Requirements
Requires `lonboard` (installed via the `viz-fast` option of the openplaces `conda` environment).

### Fullscreen Visualization
To view the map interactively fullscreen:
1. Install `voila` into your `conda` environment.
2. From the repository root, run:
   ```bash
   voila notebooks/09_show/03_show_land_and_building_values_3d.ipynb
   voila notebooks\09_show\03_show_land_and_building_values_3d.ipynb
   ```
3. Press **F11** for browser fullscreen.

# Configure

In [ ]:
import argparse
import sys
import webbrowser
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from lonboard import Map

from openplaces.io.transform import convert_area_unit
from openplaces.viz import (
    get_admin_boundary_layer,
    get_basemap_layer,
    show_value_terrain_layer,
)
from openplaces.viz.axes import add_log_ticks
from openplaces.viz.colors import DIVERGING_COLORMAPS, get_diverging_colormap

try:
    from IPython import get_ipython

    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    IN_NOTEBOOK = False

In [ ]:
# Voila tags every cell's DOM element with a `celltag_<tag>` CSS class
# (see nbconvert's celltags.j2 macro) but ships no CSS rule to act on
# it, and its progressive-rendering path (notebook_renderer.py) never
# runs nbconvert's TagRemovePreprocessor -- so
# `--TagRemovePreprocessor.remove_cell_tags=...` does nothing under
# Voila (verified against a live `voila` server; an earlier version of
# this notebook incorrectly relied on that flag). This cell supplies
# the missing rule directly, so every `remove-cell`-tagged cell below
# is actually hidden with a plain `voila <notebook>` invocation -- no
# CLI flag needed. Left untagged (must always render for the page to
# get this rule at all); a <style> element has no visible box of its
# own, so that doesn't clutter the page even though it's technically
# "shown".
if IN_NOTEBOOK:
    from IPython.display import HTML, display

    display(HTML('<style>.celltag_remove-cell{display:none !important;}</style>'))

In [ ]:
parser = argparse.ArgumentParser(
    description='3D value terrain (parcels + buildings) for one or more admin units'
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to visualize (e.g. "US-MA-SU US-MA-MI US-MA-NO")',
    nargs='*',
    default=['US-MA-SU'],
)
parser.add_argument(
    '--missing',
    help='How to handle admin units with no processed output for a recipe',
    default='warn',
    choices=['raise', 'warn', 'ignore'],
)
parser.add_argument(
    '--unit_system',
    help="Display units: 'imperial' (land $/ac, buildings $/sqft) or "
    "'metric' (land $/ha, buildings $/m2). Only affects color/value_range "
    'display -- height always stays on a common $/m2 basis internally.',
    default='metric',
    choices=['imperial', 'metric'],
)
parser.add_argument(
    '--land_cmap',
    help='Colormap name (custom or matplotlib standard) for parcels',
    default='turbo',
)
parser.add_argument(
    '--building_cmap',
    help='Colormap name (custom or matplotlib standard) for buildings',
    default='turbo',
)
parser.add_argument(
    '--land_brightness',
    help='HSV brightness multiplier applied to the parcels colormap',
    type=float,
    default=1,
)
parser.add_argument(
    '--building_brightness',
    help='HSV brightness multiplier applied to the buildings colormap',
    type=float,
    default=1,
)
parser.add_argument(
    '--basemap',
    help='Ground-plane basemap style',
    default='positron',
    choices=['satellite', 'osm', 'positron', 'dark_matter', 'topo'],
)
# vmin/vmax are manually tuned display bounds, given in $/ac (land) /
# $/sqft (buildings), then converted to whichever --unit_system
# resolves to (see the config cell below).
parser.add_argument(
    '--land_vmin',
    help='Color-scale lower bound for parcels, in $/ac',
    type=float,
    default=250,
)
parser.add_argument(
    '--land_vmax',
    help='Color-scale upper bound for parcels, in $/ac',
    type=float,
    default=10_000_000,
)
parser.add_argument(
    '--building_vmin',
    help='Color-scale lower bound for buildings, in $/sqft',
    type=float,
    default=25,
)
parser.add_argument(
    '--building_vmax',
    help='Color-scale upper bound for buildings, in $/sqft',
    type=float,
    default=50_000,
)

# Test arguments

In [ ]:
args_test = (
    # '--admin_ids US-MA-ES-AN US-MA-ES-BE US-MA-ES-DA US-MA-ES-ES US-MA-ES-GL US-MA-ES-HA US-MA-ES-LN US-MA-ES-LY US-MA-ES-MA US-MA-ES-MB US-MA-ES-MI US-MA-ES-NA US-MA-ES-NH US-MA-ES-PE US-MA-ES-SA US-MA-ES-SU US-MA-ES-SW US-MA-ES-TO US-MA-ES-WE US-MA-MI-AC US-MA-MI-AH US-MA-MI-AR US-MA-MI-BE US-MA-MI-BI US-MA-MI-BL US-MA-MI-BU US-MA-MI-CA US-MA-MI-CH US-MA-MI-CO US-MA-MI-CR US-MA-MI-EV US-MA-MI-FR US-MA-MI-HO US-MA-MI-LE US-MA-MI-LI US-MA-MI-LO US-MA-MI-MA US-MA-MI-ME US-MA-MI-ML US-MA-MI-NA US-MA-MI-NE US-MA-MI-NR US-MA-MI-RE US-MA-MI-SH US-MA-MI-SO US-MA-MI-ST US-MA-MI-SU US-MA-MI-TE US-MA-MI-WA US-MA-MI-WE US-MA-MI-WI US-MA-MI-WL US-MA-MI-WN US-MA-MI-WO US-MA-MI-WS US-MA-MI-WT US-MA-MI-WY US-MA-NO-AV US-MA-NO-BR US-MA-NO-BT US-MA-NO-CA US-MA-NO-CO US-MA-NO-DE US-MA-NO-DO US-MA-NO-FT US-MA-NO-HO US-MA-NO-MD US-MA-NO-ME US-MA-NO-MI US-MA-NO-ML US-MA-NO-NE US-MA-NO-NO US-MA-NO-NR US-MA-NO-QU US-MA-NO-RT US-MA-NO-SH US-MA-NO-ST US-MA-NO-WA US-MA-NO-WE US-MA-NO-WS US-MA-NO-WT US-MA-PL-AB US-MA-PL-HI US-MA-PL-HN US-MA-PL-HU US-MA-PL-MR US-MA-PL-NO US-MA-PL-RC US-MA-PL-SC US-MA-SU-BO US-MA-SU-CH US-MA-SU-RE US-MA-SU-WT '
    # '--admin_ids US-NC-CE '  # US-NC-BS '
    '--admin_ids US-PA-DA '
    # '--admin_ids US-NC-CE US-NC-ON US-NC-PD US-NC-NE US-NC-BS '
    # '--admin_ids US-MA-MI '
    # US-MA-NO US-MA-SU '
    # '--admin_ids US-MA-MI US-MA-NO US-MA-PL US-MA-SU '
    # "--missing warn "
    # "--unit_system imperial "
    '--land_cmap turbo '
    '--building_cmap jet '
    # "--land_brightness 1.2 "
    # "--building_brightness 0.8 "
    # "--land_vmin 1000 "
    # "--land_vmax 10000000 "
    '--building_vmin 10 '
    '--building_vmax 2500 '
    '--basemap dark_matter '
)

if IN_NOTEBOOK:
    args_list = [x for x in args_test.split(' ') if x]
else:
    args_list = sys.argv[1:] or [x for x in args_test.split(' ') if x]

args = parser.parse_args(args_list)
args

In [ ]:
def show_colormap_swatches(names=None, width=6, swatch_height=0.35):
    """Preview colormaps as horizontal gradient swatches."""
    if names is None:
        names = list(DIVERGING_COLORMAPS) + [
            'turbo',
            'jet',
            'nipy_spectral',
            'terrain',
            'rainbow',
            'pride',
            'CET_R1',
            'CET_R4',
        ]
    else:
        names = list(names)
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    fig, axes = plt.subplots(len(names), 1, figsize=(width, swatch_height * len(names)))
    for ax, name in zip(axes, names, strict=True):
        ax.imshow(gradient, aspect='auto', cmap=get_diverging_colormap(name))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha='right', va='center', fontsize=9)
    fig.tight_layout()
    return fig


_ = show_colormap_swatches()

# Configure display

In [ ]:
# Map unit systems to display area units for color scaling and
# colorbars. Extrusion height remains physically scaled in $/m2
# internally.
unit_system = args.unit_system

_AREA_UNITS = {
    'imperial': {'land': 'ac', 'building': 'sqft'},
    'metric': {'land': 'ha', 'building': 'm2'},
}
land_area_unit = _AREA_UNITS[unit_system]['land']
building_area_unit = _AREA_UNITS[unit_system]['building']

# Resolve colormaps (supports both custom palette names and standard
# matplotlib colormaps).
building_cmap = get_diverging_colormap(args.building_cmap)
land_cmap = get_diverging_colormap(args.land_cmap)

# Rescale default color bounds (defined in $/ac for land and $/sqft
# for buildings) to match the active display units selected by
# --unit_system.
land_vmin = convert_area_unit(args.land_vmin, 'ac', land_area_unit)
land_vmax = convert_area_unit(args.land_vmax, 'ac', land_area_unit)
building_vmin = convert_area_unit(args.building_vmin, 'sqft', building_area_unit)
building_vmax = convert_area_unit(args.building_vmax, 'sqft', building_area_unit)

In [ ]:
# Above this many total features, Map.to_html() embeds the entire
# dataset inline in one static HTML file (Lonboard's static export has
# no live kernel to stream from) -- for a map this size that's a 700+
# MB file no browser can practically render, so warn and skip it
# rather than write out something unusable. See Voila instead (intro
# markdown, "Viewing this map fullscreen").
_TO_HTML_FEATURE_LIMIT = 50_000


def show_or_save(m, filename):
    """Render inline in a notebook; outside one, save HTML and open it."""
    if IN_NOTEBOOK:
        return m
    # BitmapTileLayer (the basemap) has no `.table` -- it's a URL tile
    # source, not a features layer -- so skip it via getattr.
    n_features = sum(
        len(layer.table)
        for layer in m.layers
        if getattr(layer, 'table', None) is not None
    )
    if n_features > _TO_HTML_FEATURE_LIMIT:
        print(
            f'{n_features:,} total features across {len(m.layers)} layer(s) -- '
            f'over the {_TO_HTML_FEATURE_LIMIT:,}-feature limit for a static '
            'HTML export (Map.to_html() embeds the entire dataset inline and '
            "won't render at this size). Skipping the export -- use Voila "
            'instead (see the intro markdown, "Viewing this map fullscreen").'
        )
        return None
    path = Path(filename).resolve()
    m.to_html(str(path), title=filename)
    webbrowser.open(f'file://{path}')
    print(f'Saved and opened: {path}')
    return None

# Build map

## Parcels

In [ ]:
# Split into two cells (parcels, then buildings) so Voila's
# "Executing N of M" progress counter reflects which one is still
# running instead of freezing on a single combined step for the whole
# load+build pipeline -- Voila has no built-in per-cell
# description/timing display, just this generic counter (checked its
# template source: `voila_setup.macro.html.j2` hardcodes "Executing
# ${cell_index} of ${cell_count}"), so finer-grained cells is the
# practical way to get more informative progress out of it.
#
# height_clip_percentile=99.9999 is a manually tuned value, well above
# the function's own default of 99.9 -- effectively leaves nearly all
# rows uncapped for this admin selection. Re-check it before reusing
# on a different one.
# elevation_scale isn't passed explicitly here -- both calls share
# DEFAULT_ELEVATION_SCALE (from openplaces.viz.terrain) by default,
# which is what actually keeps $/m^2 -> height consistent between land
# and building value, not the cell split itself.
parcels_3d = show_value_terrain_layer(
    'US_parcel-openplaces-2026',
    args.admin_ids,
    value_column='land_value_imputed',  # real, else the street-wise imputed estimate
    area_unit=land_area_unit,
    vmin=land_vmin,
    vmax=land_vmax,
    height_clip_percentile=99.9999,
    cmap=land_cmap,
    brightness=args.land_brightness,
    outline_width=3,
    alpha=0.9,
    missing=args.missing,
)
print(
    f'{len(parcels_3d.layer.table)} parcels, elevation_scale={parcels_3d.elevation_scale:.6g}'
)

## Admin layer

In [ ]:
# Display admin-4 boundaries as a translucent 3D wall fence.
admin4_layer = get_admin_boundary_layer(
    args.admin_ids,
    level=4,
    recipe='US_admin-census-2021_admin4',
    elevation=5,
    width=1,
    color='magenta',
    opacity=0.75,
    mode='fence',
    fill_color='magenta',
    fill_opacity=0.2,
)

## Buildings

In [ ]:
# stack_on=parcels: places each building on top of the parcel
# underneath it -- requires parcels_3d (previous cell) to already be
# built, so this cell must run second; that's also why it's a separate
# cell rather than merged back with parcels_3d above (see that cell's
# comment re: Voila progress).
# value_column='structure_value' is the footprint-entity coalesce
# (apportion_curated_values, from US_parcel-openplaces-2026's already-
# coalesced improvement_value_imputed, named to match NSI's own
# structure_value_building_nsi): the real improvement value, or (real
# total minus the imputed land share) on parcels impute_land_value
# actually estimated a land value for.
buildings_3d = show_value_terrain_layer(
    'US_footprint-cheer-2026',
    args.admin_ids,
    value_column='structure_value',  # real, else (real - imputed land share)
    area_unit=building_area_unit,
    vmin=building_vmin,
    vmax=building_vmax,
    # clipped_fill_rgba=(255, 0, 255, 63),
    height_clip_percentile=99.999,
    cmap=building_cmap,
    brightness=args.building_brightness,
    outline_width=0,
    stack_on=parcels_3d,
    alpha=0.9,
    missing=args.missing,
)
print(
    f'{len(buildings_3d.layer.table)} buildings, '
    f'elevation_scale={buildings_3d.elevation_scale:.6g}'
)

# Show colorbar

In [ ]:
def show_log_colorbar(value_range, cmap, unit_suffix, subs=(1, 3)):
    """Colorbar matching a `show_value_terrain_layer` call's color ramp."""
    vmin, vmax = value_range
    norm = mpl.colors.Normalize(vmin=np.log1p(vmin), vmax=np.log1p(vmax))
    fig, ax = plt.subplots(figsize=(10, 1.4))
    cbar = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='horizontal')
    # short_number's own `sep` separates the number from ITS K/M/G
    # suffix; our per-area unit is a separate `suffix` appended after
    # that.
    add_log_ticks(
        cbar.ax,
        transform=np.log1p,
        axis='x',
        prefix='$',
        sep='',
        subs=subs,
        suffix=unit_suffix,
    )
    cbar.ax.tick_params(axis='x', rotation=90, labelsize=8)
    fig.tight_layout()
    return fig


# Use the same (land_vmin, land_vmax)/(building_vmin, building_vmax)
# bounds actually passed to show_value_terrain_layer above -- not
# parcels_3d.value_range/buildings.value_range (each layer's raw
# observed min/max), which is a *wider* range now that color is pinned
# to these manually tuned vmin/vmax constants. Using value_range here
# would draw a colorbar that doesn't match the colors actually
# rendered on the map.
# subs=(1,) (decades only) for the wider parcel range keeps it
# legible; the narrower building range can afford subs=(1, 3).
# (Assigned to `_` rather than left as the last expression, so only
# one auto-displayed figure per call shows up -- not an extra repr of
# whichever call happens to be last.)
_ = show_log_colorbar(
    (building_vmin, building_vmax), building_cmap, f'/{building_area_unit}', subs=(1, 3)
)
_ = show_log_colorbar(
    (land_vmin, land_vmax), land_cmap, f'/{land_area_unit}', subs=(1,)
)

# Show map

In [ ]:
from openplaces import get_admin

get_admin('US-PA-DA', geom=True).centroid

In [ ]:
from lonboard.view_state import MapViewState

view_state = MapViewState(
    longitude=-76.77,
    latitude=40.41,
    zoom=13,
    pitch=75,
    bearing=0,
    max_pitch=85,
)

In [ ]:
# Basemap floor first, then land (fill [+ outline]), then buildings_3d
# on top (fill [+ outline]), then floating admin-4 boundaries --
# outline_layer/clipped_layer are None whenever outline_width=0 / no
# row was actually clipped (parcels_3d above uses outline_width=0), so
# both are added conditionally rather than assumed present. admin4_layer
# is appended last so deck.gl draws floating outlines over all 3D
# surface features.
layers = [get_basemap_layer(args.basemap), parcels_3d.layer]
if parcels_3d.outline_layer is not None:
    layers.append(parcels_3d.outline_layer)
if parcels_3d.clipped_layer is not None:
    layers.append(parcels_3d.clipped_layer)
layers.append(buildings_3d.layer)
if buildings_3d.outline_layer is not None:
    layers.append(buildings_3d.outline_layer)
if buildings_3d.clipped_layer is not None:
    layers.append(buildings_3d.clipped_layer)
layers.append(admin4_layer)

map_widget = Map(layers, height='100vh', view_state=view_state)
if IN_NOTEBOOK:
    # A drag handle on the widget's own bottom edge, independent of
    # height=: lets you resize it further by hand (mouse-drag) without
    # editing code, on top of Ctrl/Cmd +/- browser zoom and Lonboard's
    # own FullscreenControl button (both already work natively -- see
    # the intro markdown). resize="vertical" only exposes the
    # bottom-edge handle, not the corner one that would also affect
    # width.
    map_widget.layout.resize = 'vertical'
    map_widget.layout.overflow = 'hidden'

show_or_save(map_widget, 'terrain_3d.html')

# Inspect data

In [ ]:
from openplaces.api import get_entities

parcels = get_entities('US_parcel-openplaces-2026', args.admin_ids, geom=True)
parcels.sample(5).T

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
# from openplaces.flow import convert_to_script

# COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

# convert_to_script(commit=COMMIT)

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)